# Introduction

In [ ]:
from revpy import reversible, ReversibleFunction

add = ReversibleFunction(lambda x, a: x + a, lambda x, a: x - a)

@reversible
def twice(x):
    y = add(1)[x]
    z = add(1)[y]
    return z

twice[0]      # -> 2 = twice(0)
twice.rev[2]  # -> 0 — the result of the automatically generated code

0

# `ReversibleFunction`: basic reversible functions

In [2]:
from revpy import ReversibleFunction, reversibility_assert
import numpy as np

move = ReversibleFunction(
    f=lambda x: x,
    f_rev=lambda x: x,
)

add = ReversibleFunction(
    f=lambda x, a: x + a, 
    f_rev=lambda x, a: x - a
)

mult = ReversibleFunction(
    f=lambda x, a: x * a, 
    f_rev=lambda x, a: x / a
)


def init(none, value):
    reversibility_assert(none is None)
    return value

def destroy(x, value):
    reversibility_assert(x == value)
    return None

init = ReversibleFunction(init, destroy)
destroy = init.rev


def init_float(none, value, **kwargs):
    reversibility_assert(none is None)
    return value

def destroy_float(x, value, **kwargs):
    reversibility_assert(np.isclose(x, value, **kwargs).all())
    return None

init_float = ReversibleFunction(init_float, destroy_float)
destroy_float = init_float.rev

# `@reversible`

## `if/fi`

In [ ]:
# если condition верно, то вернёт a * x + b
# иначе вернёт x + 1

@reversible
def f(x, a, b, condition: bool):
    
    if condition:
        x = mult(a)[x]
        x = add(b)[x]    
    else:
        x = add(1)[x]
    fi(condition)   # опционально
    return x

f(a=2, b=3, condition=True)[1.0]     # -> 5.0, f(1.0, 2, 3, True)
f(a=2, b=3, condition=True).rev[5.0] # -> 1.0, the result of the automatically generated code

f(a=2, b=3, condition=False)[1.0]     # -> 2.0, f(1.0, 2, 3, False)
f(a=2, b=3, condition=False).rev[5.0] # -> 4.0, the result of the automatically generated code

4.0

In [14]:
print(f.src)

def f(x, a, b, condition: bool):
    if condition:
        x = mult(a)[x]
        x = add(b)[x]
        reversibility_assert(condition, "reversibility_assert is False: 'condition'")
    else:
        x = add(1)[x]
        reversibility_assert(not condition, "reversibility_assert is False: 'not condition'")
    return x


In [12]:
print(f.rev.src)

def f_rev(x, a, b, condition: bool):
    if condition:
        x = add(b).rev[x]
        x = mult(a).rev[x]
        reversibility_assert(condition, "reversibility_assert is False: 'condition'")
    else:
        x = add(1).rev[x]
        reversibility_assert(not condition, "reversibility_assert is False: 'not condition'")
    return x


## `while/elihw`

In [ ]:
@reversible
def add_n_times(x, a, n=10):
    i = init(0)[None]
    while i < n:
        x = add(a)[x]
        i = add(1)[i]
    elihw(i > 0)
    destroy(n)[i]
    return x

add_n_times(a=1, n=10)[1]   # -> 11 = add_n_times(1, a=1, n=10)
add_n_times(a=1).rev[11]    # -> 1, the result of the automatically generated code

1

In [7]:
print(add_n_times.src)

def add_n_times(x, a, n=10):
    i = init(0)[None]
    reversibility_assert(not i > 0, "reversibility_assert is False: 'not i > 0'")
    while i < n:
        x = add(a)[x]
        i = add(1)[i]
        reversibility_assert(i > 0, "reversibility_assert is False: 'i > 0'")
    reversibility_assert(destroy(n)[i] is None, "reversibility_assert is False: 'destroy(n)[i] is None'")
    return x


In [8]:
print(add_n_times.rev.src)

def add_n_times_rev(x, a, n=10):
    i = destroy(n).rev[None]
    reversibility_assert(not i < n, "reversibility_assert is False: 'not i < n'")
    while i > 0:
        i = add(1).rev[i]
        x = add(a).rev[x]
        reversibility_assert(i < n, "reversibility_assert is False: 'i < n'")
    reversibility_assert(init(0).rev[i] is None, "reversibility_assert is False: 'init(0).rev[i] is None'")
    return x


### Прочие строки кода

In [ ]:
@reversible
def add_n_times(x, a, n=10):
    i = init(0)[None]
    while i < n:
        x = add(a)[x]
        i = add(1)[i]
        print(x, end=' ')
    elihw(i > 0)
    destroy(n)[i]
    return x

print(f"result={add_n_times(a=1)[0]}")      # forward call
print(f"result={add_n_times(a=1).rev[10]}") # backward call

1 2 3 4 5 6 7 8 9 10 result=10
10 9 8 7 6 5 4 3 2 1 result=0
